# Load Pips

Data set link - 
https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset

In [3]:

# 1 Import Required Libraries

import matplotlib.pyplot as plt        # For plotting graphs (accuracy/loss)
import seaborn as sns                  # For advanced visualizations (heatmap)
import pandas as pd                    # For handling dataset
import tensorflow as tf                # Main deep learning framework
from tensorflow import keras           # High-level API


# Image Data Processing

In [4]:
train_data = keras.utils.image_dataset_from_directory(
    'train',
    label_mode='categorical',   # 38 classes → one-hot encoding
    image_size=(224, 224),
    batch_size=32,
    shuffle=True
)

valid_data = tf.keras.utils.image_dataset_from_directory(
    'valid',                    # Path to validation folder
    label_mode='categorical',   # Multi-class (38 classes → one-hot)
    image_size=(224, 224),      # Resize images
    batch_size=32,              # 32 images per batch
    shuffle=False               # No need to shuffle validation data
)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'train'

EarlyStop

In [ ]:
# Import EarlyStopping class from Keras callbacks module
from keras.callbacks import EarlyStopping


# Create EarlyStopping object and store in variable 'callback'
callback = EarlyStopping(

    monitor='val_loss',        # Metric to monitor (validation loss)

    min_delta=0.01,            # Minimum change required to count as improvement

    patience=5,                # Number of epochs to wait before stopping if no improvement

    verbose=1,                 # Print message when early stopping happens (1 = show message)

    mode='auto',               # Automatically decide min/max based on monitored metric

    baseline=None,             # Training will stop if model doesn't reach this baseline value (None = ignore)

    restore_best_weights=True, # After stopping, restore weights from best epoch

    start_from_epoch=0         # Start monitoring from this epoch number
)

# Build CNN

In [6]:
# models
from keras.models import Sequential   # model
from keras.layers import (Dense, Conv2D, Dropout, 
                          BatchNormalization, 
                          Flatten, MaxPool2D,Concatenate,GlobalAveragePooling2D) # Layer
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras import layers, models

In [7]:
CNN = Sequential()

# Layer 1
CNN.add(Conv2D(32, 3, padding='same', activation='relu', input_shape=(128,128,3)))

# Layer 2
CNN.add(Conv2D(32, 3, padding='same', activation='relu'))

# Layer 3
CNN.add(Conv2D(64, 3, padding='same', activation='relu'))
CNN.add(MaxPool2D(2,2))
CNN.add(Dropout(0.15))
CNN.add(BatchNormalization())

# Layer 4
CNN.add(Conv2D(64, 3, padding='same', activation='relu'))
CNN.add(MaxPool2D(2,2))
CNN.add(Dropout(0.2))
CNN.add(BatchNormalization())

# ✅ YOUR NOVELTY (GAP)
CNN.add(GlobalAveragePooling2D())

# Dense
CNN.add(Dense(1500, activation='relu'))
CNN.add(Dropout(0.4))

# Output
CNN.add(Dense(39, activation='softmax'))  

c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Compile

In [ ]:
CNN.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Train

In [ ]:
CNN_history = CNN.fit(
    train_data,                 # Training dataset
    validation_data=valid_data, # Validation dataset
    epochs=1,                  # Number of training cycles
    verbose=1,                  # Show training progress
    callbacks=[callback]        # Custom callbacks (like EarlyStopping)
)

Accuracy

In [ ]:
#Model Evaluation on Training set
train_loss,train_acc = CNN.evaluate(train_data)

val_loss,val_acc = CNN.evaluate(valid_data)

# Save Model

In [ ]:
#Recording History in json
import json
with open("training_hist.json","w") as f:
    json.dump(CNN_history.history,f)

In [ ]:
model.load_weights("leaf_1.keras", skip_mismatch=True)

# Evalution

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf

# Load model (already trained)
model = tf.keras.models.load_model("your_model.keras", compile=False)

# Load your test data (example)
# X_test → images
# y_test → one-hot encoded labels

# 🔹 Predict
y_pred_probs = CNN.predict(valid_data)

# 🔹 Convert probabilities → class index
y_pred = np.argmax(y_pred_probs, axis=1)

# 🔹 Convert one-hot → class index
y_true = np.argmax(valid_data, axis=1)

# 🔹 Optional: class names (important for readability)
class_name = valid_data.class_names


# 🔹 Classification Report
print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_names))

# 🔹 Confusion Matrix
print("=== Confusion Matrix ===")
print(confusion_matrix(y_true, y_pred))